# Optional Vision 01 - One detector, many places

A vertical bar can move sideways without becoming horizontal. Flattening
pixels into unrelated columns makes a network relearn that local pattern
at different addresses. Can we reuse the same detector instead?

**Guiding question:** What changes when weights are shared across space,
and how does feedback reach through deeper chains?

This branch deliberately moves from sensor columns to **local 16×16
grayscale bars**. You already know modules and the update loop; images
make locality, pooling, and shared filters visible. No image download or
pretrained model is needed.

| Complaint | Evidence |
|---|---|
| A shifted edge shouldn't need new weights | One fixed filter, manual/framework agreement |
| Response maps are large | Compare stride and pooling geometry |
| A hand-chosen edge rule isn't a classifier | Train shared filters on orientation labels |
| A high score hides the route | Inspect filters/maps and trace input influence |
| A long chain can shrink backward feedback | Matched transforms with/without identity shortcuts |

We will train on the local generator and check new examples from it.
That isn't a real-photograph benchmark, transfer learning, or detection.
The branch is optional for the language route.
[Handwritten theory](01-pytorch-cnns-theory.md).


## Give a shift the same answer

The generator alternates vertical label 0 and horizontal label 1, moves
bars by small offsets, adds noise, and clips pixels to `[0,1]`.
It creates 320 practice images and 80 validation images.

```text
one image     [16,16]       pixels, not features named by hand
stack images  [320,16,16]   add an example axis
add channel   [320,1,16,16] one grayscale channel per example
labels        [320]        one orientation ID per example
```

The channel axis is not another image axis. The always-vertical baseline
ignores every pixel and gets half this balanced validation set right.
The next cell prints that baseline so a correctly shaped classifier
doesn't receive credit merely for producing labels.


In [ ]:
import sys
print(f"Kernel interpreter: {sys.executable}")

# -- Setup and local shape generator ---------------------------------------------
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white"})

def make_shape(label, size=16, jitter=True):
    image = torch.zeros(size, size)
    shift_x = int(torch.randint(-2, 3, ()).item()) if jitter else 0
    shift_y = int(torch.randint(-2, 3, ()).item()) if jitter else 0
    center = size // 2
    if label == 0:
        column = max(2, min(size - 3, center + shift_x))
        image[2:size - 2, column - 1:column + 1] = 1.0
    else:
        row = max(2, min(size - 3, center + shift_y))
        image[row - 1:row + 1, 2:size - 2] = 1.0
    image += 0.08 * torch.randn_like(image)
    return image.clamp(0, 1)

# 320 separate (16 rows, 16 columns) images -> (320, 16, 16) -> (320, 1, 16, 16).
# Stack creates an example axis; unsqueeze adds the one grayscale channel Conv2d expects.
train_images = torch.stack([make_shape(index % 2) for index in range(320)]).unsqueeze(1)
train_labels = torch.tensor([index % 2 for index in range(320)])
# The 80 validation images use the same (examples, channel, height, width) layout.
valid_images = torch.stack([make_shape(index % 2) for index in range(80)]).unsqueeze(1)
valid_labels = torch.tensor([index % 2 for index in range(80)])
print(f"Training tensor: {tuple(train_images.shape)}")
# (80,) correctness flags -> scalar: an always-vertical guess ignores image evidence.
baseline_accuracy = (valid_labels == 0).float().mean().item()
print(f"Always-vertical baseline: {baseline_accuracy:.1%}")


## 1 - Build one map value before sliding everywhere

![The same local edge detector visits shifted vertical bars; each local patch and shared kernel combine into one response-map value.](../../images/vision-sliding-detector.png)

A **filter** is a small table of reusable weights. At one location,
multiply its entries with the matching pixel patch and add. Move the
same weights, repeat: the resulting grid is a **feature map**.

The fixed Sobel filter below has rows `[-1,0,1]`, `[-2,0,2]`,
`[-1,0,1]`. It responds to changes from left to right—edges of a vertical
bar. For a clean illustrative patch whose three rows are `[0,0,1]`,
the contributions sum to `1+2+1=4`. The actual noisy patch below has
its **own printed value**, which needn't be 4.

> **Visual to add — Nine products buy one response**
> Draw that clean three-by-three `[0,0,1]`-row patch over the exact Sobel
> weights, highlight the right-column contributions 1, 2, 1, and collect
> them into one feature-map square marked 4. Label it "clean arithmetic
> close-up"; contrast with the noisy patch whose value is measured in code.
> Suggested caption: "The filter is reused; each patch supplies new pixels."

The next cell compares manual multiplication with `conv2d` at output
position `(4,4)`. No padding means a 3×3 patch fits at 14×14 positions
inside a 16×16 image. PyTorch uses the displayed filter orientation
directly; no hidden flip is needed for this comparison.


In [ ]:
# -- Manual convolution pixel and PyTorch agreement ------------------------------
image = valid_images[0:1]
# One (input channel=1, 3, 3) kernel -> (output filters=1, input channels=1, 3, 3).
# conv2d needs both channel axes, even for one grayscale edge detector.
sobel_vertical = torch.tensor(
    [[[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]]
).unsqueeze(0)
patch = image[0, :, 4:7, 4:7]
# Products (1 channel, 3, 3) -> scalar: combine all local pixel contributions.
manual_pixel = (patch * sobel_vertical[0]).sum()
# (1 image, 1 channel, 16, 16) * (1 filter, 1, 3, 3) -> (1, 1, 14, 14):
# no padding means only positions where the full 3x3 patch fits get an output.
feature_map = nn.functional.conv2d(image, sobel_vertical)
framework_pixel = feature_map[0, 0, 4, 4]
print(f"Manual sliding dot product: {manual_pixel.item():.6f}")
print(f"PyTorch feature-map pixel:  {framework_pixel.item():.6f}")
print(f"Agreement: {torch.allclose(manual_pixel, framework_pixel)}")
assert torch.allclose(manual_pixel, framework_pixel)


### One matching pixel doesn't tell the whole edge story

Apply the same fixed detector to a vertical and a horizontal bar.
No filter training occurs. The plots compare both pictures and their
absolute response maps; the printout sums response magnitudes.

Taking absolute values prevents opposite edge directions cancelling.
The total isn't a probability or an orientation classifier. Both images
can respond because horizontal-bar ends, noise, and other local changes
still contain left-to-right contrast.

**Predict:** Which total is larger in this draw? Let the paired maps and
printed sums answer. Weight sharing reuses the rule, not the response value.


In [ ]:
# -- Sobel evidence: detector location changes, weights do not -------------------
# Each (16, 16) image -> (1, 1, 16, 16): add batch and grayscale axes
# so the same conv2d detector can read either orientation.
horizontal_image = make_shape(1, jitter=False).unsqueeze(0).unsqueeze(0)
vertical_image = make_shape(0, jitter=False).unsqueeze(0).unsqueeze(0)
vertical_response = nn.functional.conv2d(vertical_image, sobel_vertical).abs()
horizontal_response = nn.functional.conv2d(horizontal_image, sobel_vertical).abs()

fig, axes = plt.subplots(2, 2, figsize=(8, 7))
axes[0, 0].imshow(vertical_image[0, 0], cmap="gray")
axes[0, 1].imshow(vertical_response[0, 0], cmap="magma")
axes[1, 0].imshow(horizontal_image[0, 0], cmap="gray")
axes[1, 1].imshow(horizontal_response[0, 0], cmap="magma")
for axis in axes.flat:
    axis.axis("off")
plt.show()
# Each (1, 1, 14, 14) response -> scalar sum: total magnitude, not class probability.
print(f"Vertical-shape response sum:   {vertical_response.sum().item():.3f}")
print(f"Horizontal-shape response sum: {horizontal_response.sum().item():.3f}")


We kept a response at every valid nearby position. To carry less spatial
detail forward, either visit fewer positions or summarize a map we've
already computed. Those shrinkage operations have different jobs.


## 2 - Skip positions, or summarize their responses?

**Stride** changes how far a convolution moves between patches.
**Max-pooling** takes an existing map and keeps the largest value in each
small region. It has no learned weights.

Here a padded stride-one convolution makes four 16×16 maps. Using those
same random filters at stride two makes four 8×8 maps. Pooling the
stride-one maps in 2×2 blocks also makes 8×8 maps.

Equal shapes don't mean equal answers: one samples weighted patch
calculations more sparsely; the other selects local maxima from a
computed map. The next cell checks geometry, not numerical equivalence
or learned detection quality.


In [ ]:
# -- Stride and pooling shape evidence -------------------------------------------
random_filter = torch.randn(4, 1, 3, 3)
# (1, 1, 16, 16) -> (1, 4, 16, 16): four filters, with padding keeping space.
stride_one = nn.functional.conv2d(image, random_filter, padding=1, stride=1)
# Same input -> (1, 4, 8, 8): stride two samples fewer patch positions.
stride_two = nn.functional.conv2d(image, random_filter, padding=1, stride=2)
# (1, 4, 16, 16) -> (1, 4, 8, 8): summarize each map's 2x2 blocks;
# unlike strided convolution, pooling combines no learned channel weights.
pooled = nn.functional.max_pool2d(stride_one, kernel_size=2)
print(f"Stride 1: {tuple(stride_one.shape)}")
print(f"Stride 2: {tuple(stride_two.shape)}")
print(f"Pool after stride 1: {tuple(pooled.shape)}")
assert stride_two.shape == pooled.shape == (1, 4, 8, 8)
print("Takeaway: both shrink space; pooling summarizes an existing map.")


## 3 - Let orientation mistakes teach the shared rules

![The shape CNN maps one-channel images through six filters, pooling, ten filters and spatial averaging to two vertical-or-horizontal scores.](../../images/vision-shape-cnn.png)

Read the next long block as a path for **one batch**:

```text
[B,1,16,16] -> six learned maps [B,6,16,16]
            -> pool [B,6,8,8] -> ten learned maps [B,10,8,8]
            -> mean over space [B,10] -> orientation scores [B,2]
```

The second convolution mixes all six input channels for each new map.
Its filters aren't ten independent looks at raw pixels. Spatial averaging
then asks how much each final detector responded across the image, not
exactly where it responded.

The loader delivers 32 images and integer orientation labels per update.
Cross-entropy receives raw scores. Backward adjusts both convolutions and
the final linear head; Adam carries its own update memory.

The block measures untrained accuracy, runs 30 epochs, and checks trained
validation accuracy above 0.90 on this generator. `losses` records only
the **last batch** of each epoch, not an epoch-wide mean.
This model has no dropout or batch normalization, so the omitted mode
switch doesn't change its outputs here; adding either would require the
train/eval discipline from Part 05.


In [ ]:
# -- Small CNN and explicit training loop ----------------------------------------
class ShapeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 3, padding=1)
        self.conv2 = nn.Conv2d(6, 10, 3, padding=1)
        self.head = nn.Linear(10, 2)

    def features(self, images):
        # (B, 1, 16, 16) -> (B, 6, 16, 16): six local detectors, same pixel grid.
        first = torch.relu(self.conv1(images))
        # Pool (B, 6, 16, 16) -> (B, 6, 8, 8), then conv -> (B, 10, 8, 8).
        # Shrink space before mixing the six maps into ten higher-level maps.
        second = torch.relu(self.conv2(nn.functional.max_pool2d(first, 2)))
        return first, second

    def forward(self, images):
        _, second = self.features(images)
        # (B, 10, 8, 8) -> (B, 10): average spatial positions so the class
        # decision uses each detector's presence rather than its exact location.
        pooled = second.mean(dim=(-2, -1))
        # (B, 10 detector summaries) -> (B, 2 class scores), one row per image.
        return self.head(pooled)

model = ShapeCNN()
loader = DataLoader(TensorDataset(train_images, train_labels), batch_size=32, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
with torch.inference_mode():
    # (80, 2) -> (80,) class IDs -> scalar mean: measure the untrained decision baseline.
    initial_accuracy = (model(valid_images).argmax(dim=-1) == valid_labels).float().mean().item()
print(f"Untrained validation accuracy: {initial_accuracy:.1%}")
losses = []
for epoch in range(30):
    for inputs, labels in loader:
        logits = model(inputs)
        loss = nn.functional.cross_entropy(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    losses.append(float(loss.detach()))

with torch.inference_mode():
    # (80, 2) -> (80,) IDs -> scalar mean: one orientation decision per validation image.
    accuracy = (model(valid_images).argmax(dim=-1) == valid_labels).float().mean().item()
print(f"Validation accuracy: {accuracy:.3f}")
print(f"Final-batch loss by epoch: {losses[0]:.4f} -> {losses[-1]:.4f}")
assert accuracy > 0.90


### A filter is a rule; a map is what happened on this image

The first convolution stores six 3×3 filters, each reading one channel:
54 weight values plus six biases. Its response to one image contains
**six 16×16 maps**, or 1,536 activations. Training adjusts the small
stored rules; a different image changes the much larger response.

> **Visual to add — A tiny stencil leaves a large trace**
> Place one actual-size 3×3 stencil beside a 16×16 response sheet, then
> show six such stencil/sheet pairs with the same vertical-bar input.
> Label "54 filter weights + 6 biases" versus "1,536 activations for one
> image". Don't invent learned filter values or name channels as concepts.
> Suggested caption: "Parameters survive calls; maps depend on the input."

The next cell plots each learned first-layer filter above its map for
the same validation image. Different pictures reveal different local
calculations, not six certified human concepts or a complete explanation
of the final label.


In [ ]:
# -- Learned filters and feature maps --------------------------------------------
filters = model.conv1.weight.detach()
with torch.inference_mode():
    first_maps, second_maps = model.features(valid_images[:1])
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for index in range(6):
    axes[0, index].imshow(filters[index, 0], cmap="coolwarm")
    axes[0, index].set_title(f"filter {index}")
    axes[1, index].imshow(first_maps[0, index].detach(), cmap="magma")
    axes[1, index].set_title(f"map {index}")
    axes[0, index].axis("off")
    axes[1, index].axis("off")
plt.show()
print("Takeaway: training specializes shared filters; maps show where each responds.")


Later maps read earlier maps. A **receptive field** is the original input
region that can reach one later value. A gradient probe measures which
parts of that route are locally active for one image; it doesn't list
every possible connection under every input.


## 4 - How far back can one later value see?

Stacking local steps expands reach: a 3×3 convolution reads three pixels
across, a 2×2 pool extends that span to four, and the next 3×3 convolution
can span eight original pixels. That's the architecture's possible reach
away from boundaries, not a promise that every enclosed pixel matters equally.

The next probe copies one image, enables gradients on it, and selects the
largest-magnitude channel at the second layer's central **pre-activation**.
Backward then writes sensitivities into input pixels.

Thresholding those sensitivities produces a mask; row/column extrema
give its bounding box. The printed dimensions and `>3` assertions check
that this active route reaches farther than one original patch.
ReLU and max-pool choices can leave holes, so don't equate the bounding
box with a solid block of equally important pixels.


In [ ]:
# -- Receptive field via input gradient -------------------------------------------
probe = valid_images[:1].clone().requires_grad_(True)
# (1, 1, 16, 16) -> (1, 6, 16, 16): keep the pixel grid for the influence probe.
first = torch.relu(model.conv1(probe))
# (1, 6, 16, 16) -> (1, 6, 8, 8): each next-layer cell now reads a larger input area.
pooled = nn.functional.max_pool2d(first, 2)
# (1, 6, 8, 8) -> (1, 10, 8, 8): mix nearby pooled maps before selecting one cell.
second_pre_activation = model.conv2(pooled)
center_row = second_pre_activation.size(-2) // 2
center_column = second_pre_activation.size(-1) // 2
center_values = second_pre_activation[0, :, center_row, center_column]
# Ten channel magnitudes -> scalar index: choose one informative center response.
strongest_channel = int(center_values.abs().argmax())
center_values[strongest_channel].backward()
receptive_mask = probe.grad[0, 0].abs() > 1e-9
rows, columns = torch.where(receptive_mask)
# Active row/column coordinate vectors -> scalar extrema: bound this local influence mask.
height = int(rows.max() - rows.min() + 1)
width = int(columns.max() - columns.min() + 1)
print(f"Traced pre-activation channel: {strongest_channel}")
print(f"Measured receptive field bounding box: {height} x {width}")
assert height > 3 and width > 3


Seeing farther can require a longer calculation chain. Repeated small
transforms can make the backward signal tiny. Could the signal have an
unchanged route around each learned correction?


## 5 - Add a route, not another set of examples

![A twelve-feature input travels unchanged around a linear-and-tanh correction and the two are added, providing an additional backward route.](../../images/vision-residual-route.png)

Temporarily use one twelve-feature vector so only the shortcut changes.
Both chains have twelve linear-and-`tanh` stages with **copied weights**,
scaled to 0.25. Both receive the same input values.
The residual chain alone adds its current input around each transform.

```text
plain:     x -> small transform -> tanh -> next stage
residual:  x ------------------------+-> next stage
            -> small transform -> tanh
```

The addition preserves `[1,12]`; it doesn't concatenate into 24 features.
Summing final features gives the same kind of scalar probe in both routes.
Backward writes an input-gradient norm for each.

> **Visual to add — Twelve small corrections beside an open route**
> Show two chains of twelve matched "linear then tanh" steps, labelled
> copied weights scaled by 0.25. In the residual chain, carry each
> twelve-feature strip around its step and add the correction back at
> matching coordinates. Use thick/thin backward arrows only after the
> actual norm comparison; don't fabricate norm values.
> Suggested caption: "The shortcut gives feedback an identity route."

The next assertion compares measured norms for this deliberately difficult
plain chain. It is an influence experiment, not a trained ResNet benchmark
or a guarantee that every skip connection improves every task.


In [ ]:
# -- ResNet gradient proof: same transforms, one extra identity route ------------
torch.manual_seed(9)
plain_layers = nn.ModuleList([nn.Linear(12, 12, bias=False) for _ in range(12)])
residual_layers = nn.ModuleList([nn.Linear(12, 12, bias=False) for _ in range(12)])
with torch.no_grad():
    for plain, residual in zip(plain_layers, residual_layers):
        residual.weight.copy_(plain.weight)
        plain.weight.mul_(0.25)
        residual.weight.mul_(0.25)

x_plain = torch.randn(1, 12, requires_grad=True)
x_residual = x_plain.detach().clone().requires_grad_(True)
plain = x_plain
residual = x_residual
for plain_layer, residual_layer in zip(plain_layers, residual_layers):
    # Both branches keep (1 example, 12 features). The sum needs matching
    # coordinates so the unchanged input can bypass each learned transform.
    plain = torch.tanh(plain_layer(plain))
    residual = residual + torch.tanh(residual_layer(residual))
# Each (1, 12) final feature vector -> scalar sum: use the same backward probe.
plain.sum().backward()
residual.sum().backward()

# Each input gradient (1, 12) -> scalar norm: compare route sensitivity magnitudes.
plain_norm = x_plain.grad.norm().item()
residual_norm = x_residual.grad.norm().item()
print(f"Input gradient without shortcut: {plain_norm:.8f}")
print(f"Input gradient with shortcut:    {residual_norm:.8f}")
print(f"Measured route advantage: {residual_norm / max(plain_norm, 1e-12):,.1f}x")
assert residual_norm > plain_norm


## The short version

- A convolution reuses the same local weights at many positions; a map
  stores their responses to this input.
- A 16×16 image and unpadded 3×3 filter give a 14×14 response grid.
  Manual/framework agreement checks one value, not classification quality.
- Stride samples fewer patches; pooling summarizes an existing map.
  Matching 8×8 shapes don't make them interchangeable.
- The trained CNN maps images to two raw orientation scores. Its
  measured validation score concerns new bars from the same generator.
- A later receptive field reaches farther than the first filter.
  The active gradient mask is input-dependent, not the entire architecture.
- Matched small transforms isolate the residual path's backward effect.
  Addition preserves coordinates; it doesn't double feature width.

| Coverage | Where we stopped |
|---|---|
| Built and measured | Fixed convolution; response comparison; stride/pooling geometry; learned classifier; filters/maps; active input influence; matched shortcut gradients |
| Explained | Weight sharing, spatial averaging, parameter/activation distinction, possible versus active receptive field |
| Later | Real-photo generalization, large pretrained backbones, transfer learning, and detection need different tasks |

**Next:** [Autoencoders](02-pytorch-autoencoders.ipynb) asks for pixels back,
not merely orientation. A classifier can discard details that rebuilding needs.
